<a href="https://colab.research.google.com/github/obraztsovm/python_2nd_sem/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22lab2_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Тема**: Структуры Pandas, чтение сложных форматов, строковые операции и борьба с пропусками.

**Данные**: olist_orders_dataset.csv, olist_order_reviews_dataset.csv, olist_customers_dataset.csv.

**Задача**:
Данные из разных систем маркетплейса часто приходят с «шумом»: некорректные типы дат, пропуски в текстах отзывов и избыточная информация. Ваша цель — превратить этот набор файлов в чистый и типизированный массив данных для дальнейшего анализа.

**Задание 1**: Эффективная загрузка и типизация
Файлы заказов содержат много временных меток, которые Pandas по умолчанию считывает как строки.

**1.1 Загрузка**: Считайте olist_orders_dataset.csv, выбрав только необходимые столбцы: order_id, customer_id, order_status и все колонки со временем (timestamps).

In [ ]:

%cd /content
!git clone https://github.com/obraztsovm/python_2nd_sem.git

%cd /content/python_2nd_sem

/content
Cloning into 'python_2nd_sem'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 43 (delta 8), reused 30 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 25.01 MiB | 947.00 KiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/python_2nd_sem


In [ ]:
import pandas as pd

file_path = "/content/python_2nd_sem/dataset/olist_orders_dataset.csv"

columns_need = [
  'order_id',
  'customer_id',
  'order_status',
  'order_purchase_timestamp',
  'order_approved_at',
  'order_delivered_carrier_date',
  'order_delivered_customer_date',
  'order_estimated_delivery_date'
]

df_orders = pd.read_csv(file_path, usecols=columns_need)

print(df_orders.columns.tolist())


['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


**1.2 Типизация:** Преобразуйте все столбцы с датами в формат datetime64 за одну операцию (используйте pd.to_datetime с параметром errors='coerce').

In [ ]:
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
  df_orders[col] = pd.to_datetime(df_orders[col], errors='coerce')

print("типы данных после преобразования:")
print(df_orders.dtypes)





типы данных после преобразования:
order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


**3.1 Память**: Измерьте объем памяти, который занимает столбец order_status, и объясните, почему его выгодно перевести в тип category.

In [ ]:
print(df_orders['order_status'].dtype)

memory_object = df_orders['order_status'].memory_usage(deep=True)
print(f"сейчас с этим типом данных он занимает {memory_object} байт")


df_orders['order_status'] = df_orders['order_status'].astype('category')
memory_category = df_orders['order_status'].memory_usage(deep=True)
print(f"а теперь после преобразования занимает {memory_category}")

print(f"тобишь экономия {memory_object - memory_category}")

df_orders['order_status'] = df_orders['order_status'].astype('object')

object
сейчас с этим типом данных он занимает 5766064 байт
а теперь после преобразования занимает 100333
тобишь экономия 5665731


**Задание 2: Очистка текстовых данных и пропусков**
Используйте файл olist_order_reviews_dataset.csv. В нем много пустых полей, так как не все клиенты пишут развернутый отзыв.

**2.1 Обработка пропусков:** * Пропуски в review_comment_title заполните строкой "No title".
Пропуски в review_comment_message заполните строкой "No message".

In [ ]:
reviews_path = "/content/python_2nd_sem/dataset/olist_order_reviews_dataset.csv"
df_reviews = pd.read_csv(reviews_path)

df_reviews['review_comment_title'] = df_reviews['review_comment_title'].fillna("No title")
df_reviews['review_comment_message'] = df_reviews['review_comment_message'].fillna("No message")

print(df_reviews[['review_comment_title', 'review_comment_message']].isnull().sum())

review_comment_title      0
review_comment_message    0
dtype: int64


**2.2 Векторизованные строки:** С помощью методов .str приведите все отзывы к нижнему регистру и удалите спецсимволы (переносы строк \n и \r).

In [ ]:
df_reviews['review_comment_title'] = df_reviews['review_comment_title'].str.lower().str.replace('\n', ' ').str.replace('\r', ' ')
df_reviews['review_comment_message'] = df_reviews['review_comment_message'].str.lower().str.replace('\n', ' ').str.replace('\r', ' ')


print(df_reviews[['review_comment_title', 'review_comment_message']].sample(5))

      review_comment_title                  review_comment_message
13627             no title                              no message
85270             no title                                 atraso 
48388             no title                              no message
40346                ótimo  parabéns vendedor ótimo profissional  
59366             no title                     muito bom o produto


**2.3 Дубликаты:** Найдите и удалите дубликаты отзывов, оставив только самые свежие (по review_creation_date) для каждого order_id

In [ ]:
df_reviews['review_creation_date'] = pd.to_datetime(df_reviews['review_creation_date'], errors='coerce')# на всякий пожарный, как в первом задании
df_reviews = df_reviews.sort_values('review_creation_date', ascending=False)

df_reviews = df_reviews.drop_duplicates(subset=['order_id'], keep='first')

print(f"колличество дубликатов: {len(df_reviews) - df_reviews['order_id'].nunique()}")


колличество дубликатов: 0


**Задание 3:** Парсинг и фильтрация (Data Discovery)
Используйте файл olist_customers_dataset.csv.

**3.1 Парсинг:** Из столбца customer_city выделите первую букву и создайте новый столбец city_initial.

In [ ]:
customers_path = "/content/python_2nd_sem/dataset/olist_customers_dataset.csv"

df_customers = pd.read_csv(customers_path)

df_customers['city_initial'] = df_customers['customer_city'].str[0]

print(df_customers[['customer_city', 'city_initial']].head(10))


           customer_city city_initial
0                 franca            f
1  sao bernardo do campo            s
2              sao paulo            s
3        mogi das cruzes            m
4               campinas            c
5         jaragua do sul            j
6              sao paulo            s
7                timoteo            t
8               curitiba            c
9         belo horizonte            b


**2.Инструментарий:** С помощью метода .loc сделайте выборку всех клиентов из штата "SP" (Сан-Паулу), у которых почтовый индекс начинается на "01".

In [ ]:
sp_customers = df_customers.loc[
    (df_customers['customer_state'] == 'SP') &
    (df_customers['customer_zip_code_prefix'].astype(str).str.startswith('01'))
]

print(df_customers['customer_state'].unique())
print(f"Найдено клиентов: {len(sp_customers)}")
sp_customers.head()


['SP' 'SC' 'MG' 'PR' 'RJ' 'RS' 'PA' 'GO' 'ES' 'BA' 'MA' 'MS' 'CE' 'DF'
 'RN' 'PE' 'MT' 'AM' 'AP' 'AL' 'RO' 'PB' 'TO' 'PI' 'AC' 'SE' 'RR']
Найдено клиентов: 0


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,city_initial


**3.3 Индексация** Установите customer_id в качестве индекса и продемонстрируйте выборку конкретного клиента по его хеш-коду.

In [ ]:
df_customers = df_customers.set_index('customer_id')
print(df_customers.head())

sample_id = df_customers.index[0]
print(f"\nинформация о клиенте с ID {sample_id}:")
print(df_customers.loc[sample_id])

                                                customer_unique_id  \
customer_id                                                          
06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

                                  customer_zip_code_prefix  \
customer_id                                                  
06b8999e2fba1a1fbc88172c00ba8bc7                     14409   
18955e83d337fd6b2def6b18a428ac77                      9790   
4e7b3e00288586ebd08712fdd0374a03                      1151   
b2b6027bc5c5109e529d4dc6358b12c3                      8775   
4f2d8ab171c80ec8364f7c12e35b23ad                     13056   

                                          customer_city customer_state  \
